In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Read csv with species names for evolutionary paths
name_paths = pd.read_csv('evolutionary_paths_names.csv', index_col=0)

# Read csv with accessions for evolutionary paths
accession_paths = pd.read_csv('evolutionary_paths_accessions.csv', index_col=0)

# Read csv with info on taxonomic relationships
rank_paths = pd.read_csv('taxonomic_rank_relationships.csv', index_col=0)

In [ ]:
name_paths.shape

In [ ]:
name_paths.head(25)

In [ ]:
name_paths.index[10000]

In [ ]:
## Practice extracting the number from the end of each index

numbers = name_paths.index.str.split().str[-1].astype(int)

for i in np.arange(25)+10000:
    print(name_paths.index[i], numbers[i], i)

In [ ]:
name_paths.shape[0]

In [ ]:
## Determine which species have more than 10 representative evolutionary paths and filter them to a max of 10 per species

# Extract numbers from indices in dataframe
numbers = name_paths.index.str.split().str[-1].astype(int)

#Initialize list for indices requiring deletion
indices_to_delete = []

# Loop to determine which indices indicate more than 10 members for a given species
for i in np.arange(name_paths.shape[0]):
    if numbers[i] > 9:
        indices_to_delete.append(name_paths.index[i])

# Filter all three dataframes for the indices determined above
filter_name_paths1 = name_paths.drop(indices_to_delete)
filter_accession_paths1 = accession_paths.drop(indices_to_delete)
filter_rank_paths1 = rank_paths.drop(indices_to_delete)

In [ ]:
filter_name_paths1.shape

In [ ]:
# Save the evolutionary paths
filter_name_paths1.to_csv('filtered1_evolutionary_paths_names.csv')

# Save the accession paths  
filter_accession_paths1.to_csv('filtered1_evolutionary_paths_accessions.csv')

# Save to CSV
filter_rank_paths1.to_csv('filtered1_taxonomic_rank_relationships.csv')

In [ ]:
pd.set_option('display.max_rows', None)  # Show all rows


display(filter_name_paths1.head(15))
filter_name_paths1.tail(15)

In [ ]:
filter_rank_paths1.head()

In [ ]:
## Count how many members of each phylum are considered as the base taxa

# Extract words inside parentheses and handle edge cases
Phylum = filter_rank_paths1['Parentheses_Word'] = (
    filter_rank_paths1['Other Kingdom']
    .astype(str)  # Convert all entries to strings (handles NaN/float)
    .str.extract(r'\((.*?)\)')  # Extract first parenthetical word
    .fillna('Unknown')  # Replace NaN with "Unknown"
)



# Count unique occurrences
unique_counts = Phylum.value_counts()

print(unique_counts)

In [ ]:
4081+879+237+161+49+44+23+22+3+1

In [ ]:
## Count how many members of each phylum are considered as the base taxa
## Unfortunately I don't think this one is correct, for one all oomycetes are left off given their 'NaN' phylum column entries. Other taxa (rare/cryptic) likely affected also.

# Extract words inside parentheses and handle edge cases
Class = filter_rank_paths1['Parentheses_Word'] = (
    filter_rank_paths1['Same Phylum']
    .astype(str)  # Convert all entries to strings (handles NaN/float)
    .str.extract(r'\((.*?)\)')  # Extract first parenthetical word
    .fillna('Unknown')  # Replace NaN with "Unknown"
)



# Count unique occurrences
unique_counts = Class.value_counts()

print(unique_counts)

In [ ]:
unique_counts.sum()

In [ ]:
filter_accession_paths1.head()

In [ ]:
# Flatten all values into one list and count unique entries
total_unique_values = len(pd.unique(filter_accession_paths1.values.ravel('K')))
print(f"Total unique accessions across all columns: {total_unique_values}")

## Pick 1000 ascomycota assemblies of highly represented classes to remove, for a final list of 4500 evolutionary paths

In [ ]:
# Set random seed for reproducibility
np.random.seed(4)

# Define the classes and how many to sample from each
class_sample_sizes = {
    'Sordariomycetes': 400,
    'Eurotiomycetes': 300,
    'Dothideomycetes': 100,
    'Pichiomycetes': 100,
    'Saccharomycetes': 100
}

# Initialize a list to store the indices to be removed
indices_to_remove = []

# Extract the class information (as you already did)
filter_rank_paths1['Class'] = (
    filter_rank_paths1['Same Phylum']
    .astype(str)
    .str.extract(r'\((.*?)\)')
    .fillna('Unknown')
)

# For each target class, randomly sample the specified number of indices
for class_name, sample_size in class_sample_sizes.items():
    # Get indices of all rows with this class
    class_indices = filter_rank_paths1[filter_rank_paths1['Class'] == class_name].index
    
    # Check if we have enough members
    if len(class_indices) < sample_size:
        print(f"Warning: Only {len(class_indices)} {class_name} members available (requested {sample_size})")
        sample_size = len(class_indices)  # Adjust to available number
    
    # Randomly sample without replacement
    sampled_indices = np.random.choice(class_indices, size=sample_size, replace=False)
    indices_to_remove.extend(sampled_indices)

# Convert to a list (not a set) for pandas compatibility
indices_to_remove = list(indices_to_remove)

# Verify counts
print(f"Total indices marked for removal: {len(indices_to_remove)}")
print("Breakdown by class:")
for class_name in class_sample_sizes:
    count = sum(filter_rank_paths1.loc[indices_to_remove, 'Class'] == class_name)
    print(f"{class_name}: {count}")

In [ ]:
# Filter all three dataframes for the indices determined above
filter_name_paths2 = filter_name_paths1.drop(indices_to_remove)
filter_accession_paths2 = filter_accession_paths1.drop(indices_to_remove)
filter_rank_paths2 = filter_rank_paths1.drop(indices_to_remove)

In [ ]:
## Count how many members of each phylum are considered as the base taxa
## Unfortunately I don't think this one is correct, for one all oomycetes are left off given their 'NaN' phylum column entries. Other taxa (rare/cryptic) likely affected also.

# Extract words inside parentheses and handle edge cases
Class = filter_rank_paths2['Parentheses_Word'] = (
    filter_rank_paths2['Same Phylum']
    .astype(str)  # Convert all entries to strings (handles NaN/float)
    .str.extract(r'\((.*?)\)')  # Extract first parenthetical word
    .fillna('Unknown')  # Replace NaN with "Unknown"
)



# Count unique occurrences
unique_counts = Class.value_counts()

print(unique_counts)

In [ ]:
# Save the evolutionary paths
filter_name_paths2.to_csv('filtered2_evolutionary_paths_names.csv')

# Save the accession paths  
filter_accession_paths2.to_csv('filtered2_evolutionary_paths_accessions.csv')

# Save to CSV
filter_rank_paths2.to_csv('filtered2_taxonomic_rank_relationships.csv')